# Cart Abandonment Pattern Analysis
**Student:** Abhay Kumar  
**Program:** IBM SkillsBuild Data Analytics with AI Academic Internship Program  
**Conducted by:** BharatCares in association with AICTE

## Objective
Analyze e-commerce sessions to measure cart abandonment, compare device and channel segments, inspect cart values and stated reasons, and suggest checkout improvements.

**Dataset honesty note:** The CSV bundled with this project is a synthetic demo dataset generated to let the notebook run immediately. It is not a Kaggle download. A real Kaggle source is linked below; its raw data is event-level and needs session-level preprocessing.

## Dataset source
Kaggle: [eCommerce behavior data from multi category store](https://www.kaggle.com/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store). The dataset includes event types such as `view`, `cart`, `remove_from_cart`, and `purchase`, and can be very large. Review the dataset description and terms before downloading.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

paths = [Path('Data/cart_abandonment_demo_dataset.csv'), Path('/content/cart_abandonment_demo_dataset.csv'), Path('/content/Data/cart_abandonment_demo_dataset.csv')]
data_path = next((p for p in paths if p.exists()), None)
if data_path is None:
    try:
        from google.colab import files
        uploaded = files.upload()
        data_path = Path(next(iter(uploaded.keys())))
    except Exception as e:
        raise FileNotFoundError('Upload cart_abandonment_demo_dataset.csv or place it in the Data folder.') from e
df = pd.read_csv(data_path)
print('Dataset shape:', df.shape)
display(df.head())
display(df.isna().sum().to_frame('missing_values'))

## 1. Data cleaning

In [ ]:
df = df.drop_duplicates(subset='session_id').copy()
for col in ['cart_items','cart_value_inr','cart_abandoned','recovery_email_sent','recovered','session_duration_minutes','promo_code_used']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
for col in ['device_type','user_type','marketing_channel','product_category']:
    df[col] = df[col].fillna('Unknown')
df['cart_abandoned'] = df['cart_abandoned'].fillna(0).astype(int)
df['recovery_email_sent'] = df['recovery_email_sent'].fillna(0).astype(int)
df['recovered'] = df['recovered'].fillna(0).astype(int)
df['cart_value_inr'] = df['cart_value_inr'].fillna(df['cart_value_inr'].median())
print('Rows after deduplication:', len(df))
display(df.describe(include='all').T)

## 2. KPIs

In [ ]:
total = len(df); abandoned = int(df.cart_abandoned.sum()); completed = total-abandoned
rate = abandoned/total*100 if total else 0
email_coverage = df.loc[df.cart_abandoned.eq(1),'recovery_email_sent'].mean()*100 if abandoned else 0
recovered = int(df.recovered.sum())
kpis = pd.DataFrame({'Metric':['Total sessions','Abandoned carts','Abandonment rate (%)','Checkout completion rate (%)','Total value in abandoned carts (INR)','Abandoned carts sent recovery email (%)','Recovered carts','Recovery rate of abandoned carts (%)'], 'Value':[total,abandoned,round(rate,2),round(completed/total*100 if total else 0,2),round(df.loc[df.cart_abandoned.eq(1),'cart_value_inr'].sum(),2),round(email_coverage,2),recovered,round(recovered/abandoned*100 if abandoned else 0,2)]})
display(kpis)

## 3. Exploratory analysis

In [ ]:
device = df.groupby('device_type').cart_abandoned.mean().mul(100).sort_values(ascending=False)
channel = df.groupby('marketing_channel').cart_abandoned.mean().mul(100).sort_values(ascending=False)
display(device.rename('abandonment_rate_percent').to_frame())
device.plot(kind='bar', title='Cart Abandonment Rate by Device', ylabel='Rate (%)'); plt.xticks(rotation=0); plt.tight_layout(); plt.show()
display(channel.rename('abandonment_rate_percent').to_frame())
channel.plot(kind='bar', title='Cart Abandonment Rate by Marketing Channel', ylabel='Rate (%)'); plt.xticks(rotation=35, ha='right'); plt.tight_layout(); plt.show()
df.boxplot(column='cart_value_inr', by='cart_abandoned'); plt.title('Cart Value by Abandonment Status'); plt.suptitle(''); plt.xlabel('Abandoned (0=No, 1=Yes)'); plt.ylabel('Cart value (INR)'); plt.tight_layout(); plt.show()
reasons=df.loc[df.cart_abandoned.eq(1),'abandonment_reason'].value_counts(); display(reasons.to_frame('sessions'))
reasons.plot(kind='bar',title='Recorded Abandonment Reasons',ylabel='Sessions'); plt.xticks(rotation=35,ha='right'); plt.tight_layout(); plt.show()

## 4. Segment summaries

In [ ]:
for col in ['device_type','user_type','marketing_channel','product_category']:
    summary=df.groupby(col).agg(sessions=('session_id','count'), abandoned=('cart_abandoned','sum'), average_cart_value=('cart_value_inr','mean'))
    summary['abandonment_rate_percent']=(summary.abandoned/summary.sessions*100).round(2)
    print('\nSegment:', col); display(summary.sort_values('abandonment_rate_percent',ascending=False))
print(f'Overall abandonment rate: {rate:.2f}% ({abandoned}/{total} sessions)')
if len(device): print('Highest device segment:', device.index[0])
if len(channel): print('Highest channel segment:', channel.index[0])
if len(reasons): print('Most common reason in this demo:', reasons.index[0])

## 5. Recommendations
- Show shipping fees, taxes, and delivery estimates before payment.
- Test shorter checkout forms and clearer progress indicators.
- Check mobile speed, layout, and payment failures.
- Send permission-based cart reminders and measure incremental conversions.
- A/B test targeted offers rather than discounting every cart.

## Limitations and conclusion
The included dataset is synthetic; all observed patterns are illustrative and do not describe a real retailer. Segment differences do not prove causation. For real Kaggle data, aggregate event rows by `user_session`, identify sessions with cart events, and define completed sessions using purchase events before calculating abandonment. This notebook demonstrates data cleaning, KPI calculation, exploratory data analysis, visualization, and business recommendations.